# Pivot Tables

## 🧠 What is groupby in pandas?

### groupby:
- Splits data into groups
- Applies a function to each group
- Combines results back

This is called **`Split → Apply → Combine`**

## 📦 Basic Syntax
```python
df.groupby(by)[agg]
```

or
```python
df.groupby(by).agg(func)
```

In [1]:
import pandas as pd

data = {
    "Region": ["East", "East", "West", "West", "South"],
    "Product": ["A", "B", "A", "B", "A"],
    "Sales": [100, 150, 200, 250, 300],
    "Quantity": [1, 2, 3, 4, 5]
}

df = pd.DataFrame(data)
df

,Region,Product,Sales,Quantity
0,East,A,100,1
1,East,B,150,2
2,West,A,200,3
3,West,B,250,4
4,South,A,300,5


### 1️⃣ Basic GroupBy (SQL-style)

In [2]:
# Total Sales by Region
df.groupby(by="Region")["Sales"].sum()

Region
East     250
South    300
West     450
Name: Sales, dtype: int64

### 2️⃣ GroupBy Multiple Columns

In [3]:
df.groupby(by=["Region","Product"])["Sales"].sum()

Region  Product
East    A          100
        B          150
South   A          300
West    A          200
        B          250
Name: Sales, dtype: int64

### 3️⃣ Multiple Aggregations (VERY COMMON)

In [5]:
df.groupby("Region")["Sales"].agg([
    "sum",
    "count",
    "mean"
]
)

,sum,count,mean
Region,,,
East,250,2,125.0
South,300,1,300.0
West,450,2,225.0


### 4️⃣ GroupBy Multiple Columns + Multiple Aggregations

In [6]:
df.groupby("Region").agg({
    "Sales": ["sum", "mean"],
    "Quantity": "sum"
})

Sales        Quantity
         sum   mean      sum
Region                      
East     250  125.0        3
South    300  300.0        5
West     450  225.0        7

### 5️⃣ Reset Index (Almost Always Needed)

In [8]:
df.groupby("Region")["Sales"].sum().reset_index()

,Region,Sales
0,East,250
1,South,300
2,West,450


👉 Without this, result has Region as index, which can be annoying in reporting.

### 6️⃣ Named Aggregations (Clean & Recommended)

In [12]:
df.groupby("Region").agg(
    total_sales = ("Sales", "sum"),
    avg_sales = ("Sales", "mean"),
    total_qty = ("Quantity", "count")
).reset_index()

,Region,total_sales,avg_sales,total_qty
0,East,250,125.0,2
1,South,300,300.0,1
2,West,450,225.0,2


✅ Clean column names

✅ Best practice for production code

### 8️⃣ GroupBy + Sort

In [14]:
df.groupby("Region")["Sales"].sum().sort_values(ascending=False).reset_index()


,Region,Sales
0,West,450
1,South,300
2,East,250


### 9️⃣ GroupBy + Transform (IMPORTANT)
Add group-level metric back to original data

In [15]:
df['region_avg_sales'] = df.groupby("Region")["Sales"].transform("mean")
df

,Region,Product,Sales,Quantity,region_avg_sales
0,East,A,100,1,125.0
1,East,B,150,2,125.0
2,West,A,200,3,225.0
3,West,B,250,4,225.0
4,South,A,300,5,300.0


👉 Unlike agg, transform keeps row count same.

### 🔟 GroupBy + Filter
Keep regions with total sales > 300

In [31]:
df.groupby("Region").filter(
    lambda x : x['Sales'].sum()>300
    ).groupby("Region").agg(
    total_sales = ("Sales","sum")
).reset_index()

,Region,total_sales
0,West,450


#### 🧠 What’s happening step-by-step

##### 1️⃣ `groupby("Region").filter(...)` → **HAVING**

```python
df.groupby("Region").filter(
    lambda x: x["Sales"].sum() > 300
)
```

* Groups rows by `Region`
* Computes `SUM(Sales)` **per group**
* Keeps **only those groups** where sum > 300
* **Returns full rows**, not aggregated data

📌 This is the **pandas version of `HAVING`**


##### 2️⃣ Second `groupby("Region").agg(...)` → **SELECT + GROUP BY**

```python
.groupby("Region").agg(
    total_sales=("Sales", "sum")
)
```

* Re-groups the filtered rows
* Calculates the final aggregation you want to show

##### 3️⃣ `.reset_index()` → **Flat result (table-like)**

```python
.reset_index()
```

* Converts `Region` from index → column
* Makes the output reporting-friendly


#### 🧾 Exact SQL Equivalent

```sql
SELECT
    Region,
    SUM(Sales) AS total_sales
FROM table
GROUP BY Region
HAVING SUM(Sales) > 300;
```

✅ **One-to-one logical match**

#### ⚠️ Important Insight (Interview Gold)

> Pandas does **NOT** have a direct `HAVING` keyword.

So we simulate it using:

* `groupby().filter()` **before aggregation**
* then aggregate again

That’s why **two groupbys** appear.


#### 🔁 Alternative (More Compact, Same Logic)

You can also do:

```python
(
    df.groupby("Region")
      .agg(total_sales=("Sales", "sum"))
      .query("total_sales > 300")
      .reset_index()
)
```

##### Why this works

* Aggregate first
* Filter aggregated result
* Cleaner & faster for simple cases

📌 This is often preferred in **analytics code**.


#### 🆚 When to Use Which

| Approach             | Use When                             |
| -------------------- | ------------------------------------ |
| `groupby().filter()` | Need **row-level data** after HAVING |
| `agg() + query()`    | Only need **aggregated output**      |
| SQL-style thinking   | Interviews & BI discussions          |


#### 🧠 Final Mental Model

> **WHERE filters rows**
> **HAVING filters groups**

In pandas:

* `df[condition]` → WHERE
* `groupby().filter()` → HAVING




### 1️⃣1️⃣ GroupBy + Apply (Powerful but Dangerous)

In [32]:
df.groupby("Region").apply(lambda x: x.nlargest(1, "Sales"))

C:\Users\DELL\AppData\Local\Temp\ipykernel_1672\3912775818.py:1: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby("Region").apply(lambda x: x.nlargest(1, "Sales"))


,,Region,Product,Sales,Quantity,region_avg_sales
Region,,,,,,
East,1,East,B,150,2,125.0
South,4,South,A,300,5,300.0
West,3,West,B,250,4,225.0


⚠️ Flexible but slower — use only when agg / transform can’t solve it.

---



# 🔁 What is a Pivot Table (in pandas)?

A **pivot table**:

* Groups data
* Aggregates values
* Reshapes rows ↔ columns
* Produces summary tables for analysis

In pandas, this is done using:

```python
pd.pivot_table()
```


## 📦 Basic Syntax

```python
pd.pivot_table(
    data,
    values=None,
    index=None,
    columns=None,
    aggfunc='mean',
    fill_value=None,
    margins=False
)
```

### Key Parameters (you’ll use these 90% of the time)

| Parameter    | Meaning                                    |
| ------------ | ------------------------------------------ |
| `data`       | DataFrame                                  |
| `values`     | Column(s) to aggregate                     |
| `index`      | Rows (group by)                            |
| `columns`    | Columns (group by)                         |
| `aggfunc`    | Aggregation (`sum`, `mean`, `count`, etc.) |
| `fill_value` | Replace NaN                                |
| `margins`    | Add totals (like Excel Grand Total)        |


In [33]:
data = {
    "Region": ["East", "East", "West", "West", "South"],
    "Product": ["A", "B", "A", "B", "A"],
    "Sales": [100, 150, 200, 250, 300],
    "Quantity": [1, 2, 3, 4, 5]
}

df = pd.DataFrame(data)
df

,Region,Product,Sales,Quantity
0,East,A,100,1
1,East,B,150,2
2,West,A,200,3
3,West,B,250,4
4,South,A,300,5


### 1️⃣ Simple Pivot (Like GROUP BY)

In [35]:
pd.pivot_table(
    data=df,
    values="Sales",
    index="Region",
    aggfunc="sum"
).reset_index()

,Region,Sales
0,East,250
1,South,300
2,West,450


### 2️⃣ Pivot with Rows & Columns
Sales by Region × Product

In [39]:
pd.pivot_table(
    data=df,
    values="Sales",
    index="Region",
    columns="Product",
    aggfunc="sum"
)

Product,A,B
Region,,
East,100.0,150.0
South,300.0,NaN
West,200.0,250.0


### 3️⃣ Handling Missing Values

In [40]:
pd.pivot_table(
    data=df,
    values="Sales",
    index="Region",
    columns="Product",
    aggfunc="sum",
    fill_value=0
)

Product,A,B
Region,,
East,100,150
South,300,0
West,200,250


### 4️⃣ Multiple Aggregations (Very Important)

In [42]:
pd.pivot_table(
    df,
    values="Sales",
    index="Region",
    aggfunc=["sum", "mean", "count"]
).reset_index()

,Region,sum,mean,count
,,Sales,Sales,Sales
0,East,250,125.0,2
1,South,300,300.0,1
2,West,450,225.0,2


### 5️⃣ Multiple Value Columns

In [ ]:
pd.pivot_table(
    df,
    values=["Sales", "Quantity"],
    index="Region",
    aggfunc="sum"
).reset_index()

,Region,Quantity,Sales
0,East,3,250
1,South,5,300
2,West,7,450


### 6️⃣ Multi-Level Pivot (Advanced but Common)
Region → Product breakdown

In [45]:
pd.pivot_table(
    df,
    values="Sales",
    index=["Region", "Product"],
    aggfunc="sum"
)

Sales
Region Product       
East   A          100
       B          150
South  A          300
West   A          200
       B          250

### 7️⃣ Add Grand Totals (Excel-style)

In [46]:
pd.pivot_table(
    df,
    values="Sales",
    index="Region",
    columns="Product",
    aggfunc="sum",
    margins=True,
    margins_name="Total"
)


Product,A,B,Total
Region,,,
East,100.0,150.0,250
South,300.0,NaN,300
West,200.0,250.0,450
Total,600.0,400.0,1000


### 8️⃣ Multiple Aggregations per Column

In [47]:
pd.pivot_table(
    df,
    values=["Sales", "Quantity"],
    index="Region",
    aggfunc={
        "Sales": ["sum", "mean"],
        "Quantity": "sum"
    }
)

Quantity  Sales     
            sum   mean  sum
Region                     
East          3  125.0  250
South         5  300.0  300
West          7  225.0  450

### 9️⃣ Pivot vs GroupBy 

| `pivot_table`              | `groupby`            |
| -------------------------- | -------------------- |
| Reshapes data              | Just aggregates      |
| Excel-like                 | SQL-like             |
| Supports columns parameter | No columns           |
| Better for reporting       | Better for pipelines |


### 10️⃣ Common Mistakes ❌

- ❌ Using `pivot` instead of `pivot_table`
- ❌ Forgetting `aggfunc` when duplicates exist
- ❌ Not filling NaNs
- ❌ Overusing multi-index without flattening


### 🧠 Mental Model (Remember This)

> **Pivot Table = GroupBy + Aggregate + Reshape**

If you can imagine the Excel pivot:

* Rows → `index`
* Columns → `columns`
* Values → `values`
* Summarize by → `aggfunc`

---

